# Manual backpropagation for a toy two-item layout network (T17)

Viva evidence for Tech Spec Section 8 (Appendices A to F, K and N). A small network shaped like the
CVAE decoder turns a condition vector into the positions and rotations of **two items** (a sofa and a
coffee table). It is scored with the project's own loss terms:

* masked squared error on the positions (Sigmoid head),
* cross-entropy on the rotation logits (Softmax inside the loss),
* the penetration-depth overlap penalty used in latent optimization (Tech Spec 3.2).

Every gradient is derived by hand below, computed with NumPy, and compared with PyTorch autograd.
The overlap in the autograd version comes from `spacegen.geometry.penetration_depth`, the function the
project itself uses, so the check also confirms that the hand derivation matches the project's code.
Everything runs in float64; the pass mark is a largest difference below $10^{-6}$.

In [1]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "spacegen").is_dir())
sys.path.insert(0, str(root))  # so that `spacegen` imports when the notebook runs from notebooks/

import numpy as np
import torch
import torch.nn.functional as F

from spacegen import geometry

torch.set_default_dtype(torch.float64)
print("numpy", np.__version__, "| torch", torch.__version__)

numpy 2.4.6 | torch 2.5.1+cu121


## 1. The network and the loss

For a batch of $B$ conditions $x \in \mathbb{R}^{B \times 5}$ (room $W/8$, $D/8$, the two presence masks, a constant 1):

$$
\begin{aligned}
z_1 &= x W_1 + b_1, & h &= \mathrm{ReLU}(z_1) && \text{hidden layer, 6 units}\\
z_2 &= h W_2 + b_2, & p &= \sigma(z_2) \in (0,1)^{B \times 4} && (u_1, v_1, u_2, v_2)\\
z_3 &= h W_3 + b_3 &&\in \mathbb{R}^{B \times 8} && \text{rotation logits, read as } (B, 2, 4)
\end{aligned}
$$

The loss averages over the batch; $m_{bk}$ is 1 when item $k$ is present (Hadamard masking, Appendix N):

$$
L = \frac{1}{B} \sum_b \Big[ \sum_k m_{bk} \big( (u_{bk} - u^*_{bk})^2 + (v_{bk} - v^*_{bk})^2
      + \lambda_r \, \mathrm{CE}(z_{3,bk}, r^*_{bk}) \big) + \lambda_o \, m_{b1} m_{b2} \, \mathrm{pen}_b^2 \Big]
$$

The overlap penalty works in metres, $x_k = u_k W_b$ and $y_k = v_k D_b$, with the items' footprints
extents along x and y $(w_k, d_k)$ (the sofa faces East in about half of the samples, which swaps its
footprint), margin $\mu$ and the smoothing $\varepsilon = 10^{-6}$ of Tech Spec 3.2:

$$
\mathrm{pen} = \min(p_x, p_y), \qquad
p_x = \mathrm{ReLU}\Big(\tfrac{w_1 + w_2}{2} + \mu - \sqrt{(x_1 - x_2)^2 + \varepsilon}\Big), \qquad
p_y = \mathrm{ReLU}\Big(\tfrac{d_1 + d_2}{2} + \mu - \sqrt{(y_1 - y_2)^2 + \varepsilon}\Big)
$$

In [2]:
# Seed and scales chosen so that every path below is used: every pair of present items overlaps,
# the minimum picks the x axis in some samples and the y axis in others, and the ReLU units are
# active for some samples and not for others.
rng = np.random.default_rng(152)
B, n_in, n_hidden = 16, 5, 6
lambda_r, lambda_o, margin, eps = 1.0, 10.0, 0.05, geometry.SMOOTH_EPS

room = np.column_stack([rng.uniform(3.5, 7.0, B), rng.uniform(3.0, 6.0, B)])  # W, D (m)
mask = np.column_stack([np.ones(B), rng.uniform(size=B) < 0.7]).astype(float)  # the table is sometimes absent
turned = rng.uniform(size=B) < 0.5  # the sofa faces East in these samples, so its footprint swaps
sofa, table = np.array([2.10, 0.90]), np.array([1.00, 0.55])
footprint = np.stack([np.where(turned[:, None], sofa[::-1], sofa), np.broadcast_to(table, (B, 2))], axis=1)
x = np.column_stack([room / 8 - 0.6, mask[:, 1] - 0.5, turned - 0.5, np.ones(B)])  # centred inputs, then a 1

params = {"W1": rng.normal(0, 1.0, (n_in, n_hidden)), "b1": rng.normal(0, 0.1, n_hidden),
          "W2": rng.normal(0, 0.3, (n_hidden, 4)), "b2": np.zeros(4)}
target = rng.uniform(0.2, 0.8, size=(B, 4))  # (u1, v1, u2, v2)
rot_target = rng.integers(4, size=(B, 2))
params |= {"W3": rng.normal(0, 0.5, (n_hidden, 8)), "b3": rng.normal(0, 0.1, 8)}
{name: value.shape for name, value in params.items()}

{'W1': (5, 6), 'b1': (6,), 'W2': (6, 4), 'b2': (4,), 'W3': (6, 8), 'b3': (8,)}

In [3]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def forward(params):
    # Forward pass in NumPy; returns the loss and everything the backward pass needs.
    z1 = x @ params["W1"] + params["b1"]
    h = np.maximum(z1, 0)
    z2 = h @ params["W2"] + params["b2"]
    p = sigmoid(z2)
    z3 = (h @ params["W3"] + params["b3"]).reshape(B, 2, 4)

    position_mask = np.repeat(mask, 2, axis=1)  # (B, 4): u and v of each item share its mask
    loss_pos = (position_mask * (p - target) ** 2).sum() / B

    shifted = z3 - z3.max(axis=2, keepdims=True)  # stable softmax
    q = np.exp(shifted) / np.exp(shifted).sum(axis=2, keepdims=True)
    log_q = np.log(q)
    ce = -np.take_along_axis(log_q, rot_target[..., None], axis=2)[..., 0]  # (B, 2)
    loss_rot = lambda_r * (mask * ce).sum() / B

    meters = p.reshape(B, 2, 2) * room[:, None, :]  # (B, item, axis)
    gap = meters[:, 0] - meters[:, 1]  # x1 - x2, y1 - y2
    s = np.sqrt(gap ** 2 + eps)
    a = footprint.sum(axis=1) / 2 + margin  # (B, 2): (w1 + w2)/2 + mu, (d1 + d2)/2 + mu
    pxy = np.maximum(a - s, 0)  # (B, 2): p_x, p_y
    pen = pxy.min(axis=1)
    both = mask[:, 0] * mask[:, 1]
    loss_ov = lambda_o * (both * pen ** 2).sum() / B

    cache = dict(z1=z1, h=h, p=p, q=q, position_mask=position_mask, gap=gap, s=s, a=a, pxy=pxy,
                 pen=pen, both=both)
    return loss_pos + loss_rot + loss_ov, cache


loss, cache = forward(params)
overlapping = (cache["pen"] * cache["both"]) > 0
x_axis = overlapping & (cache["pxy"][:, 0] <= cache["pxy"][:, 1])
active = (cache["z1"] > 0).mean(axis=0)
print(f"loss {loss:.6f}")
print(f"pairs that overlap: {overlapping.sum()} of {int(cache['both'].sum())} with both items; the minimum picks "
      f"x in {x_axis.sum()} and y in {(overlapping & ~x_axis).sum()}")
print("share of samples where each hidden unit is active:", np.round(active, 2))

loss 7.850291
pairs that overlap: 13 of 13 with both items; the minimum picks x in 7 and y in 6
share of samples where each hidden unit is active: [0.44 1.   0.56 0.31 0.06 0.31]


## 2. The gradients, by hand

Backpropagation runs the forward pass in reverse, multiplying local derivatives (the chain rule).

**Positions (Appendix D, N).** The squared error gives
$\dfrac{\partial L}{\partial p_{bc}} = \dfrac{2}{B}\, m_{bc} (p_{bc} - p^*_{bc})$; an absent item contributes zero.

**Overlap (Appendix K).** With $L_o = \frac{\lambda_o}{B} \sum_b m_{b1} m_{b2}\, \mathrm{pen}_b^2$:
$\dfrac{\partial L_o}{\partial \mathrm{pen}_b} = \dfrac{2 \lambda_o}{B} m_{b1} m_{b2}\, \mathrm{pen}_b$.
The minimum passes the gradient to the smaller depth: $\partial \mathrm{pen}/\partial p_x = [p_x \le p_y]$.
For $p_x = \mathrm{ReLU}(a_x - s_x)$ with $s_x = \sqrt{g_x^2 + \varepsilon}$ and $g_x = x_1 - x_2 = (u_1 - u_2) W$:

$$
\frac{\partial p_x}{\partial g_x} = -[a_x - s_x > 0]\, \frac{g_x}{s_x}, \qquad
\frac{\partial g_x}{\partial u_1} = W, \qquad \frac{\partial g_x}{\partial u_2} = -W
$$

so the two items are pushed apart along the easier axis, by equal and opposite amounts. The same holds
for $v$ with $D$. This term is added to the position gradient: $g_p = \partial L / \partial p$.

**Sigmoid head (Appendix B).** $\sigma'(z) = \sigma(z)(1 - \sigma(z))$, so $\delta_2 = \partial L/\partial z_2 = g_p \odot p \odot (1 - p)$.

**Softmax and cross-entropy (Appendix C, N).** For each item, with $q = \mathrm{softmax}(z_{3,bk})$:
$\delta_{3,bk} = \dfrac{\lambda_r}{B}\, m_{bk}\, (q - \mathrm{onehot}(r^*_{bk}))$.

**Linear layers (Appendix A, F).** For $z = h W + b$: $\partial L/\partial W = h^\top \delta$,
$\partial L/\partial b = \sum_b \delta_b$ (the bias was broadcast over the batch), and
$\partial L/\partial h = \delta W^\top$. Both heads read $h$, so their contributions add:
$\partial L/\partial h = \delta_2 W_2^\top + \delta_3 W_3^\top$.

**ReLU (Appendix E).** $\delta_1 = \partial L/\partial h \odot [z_1 > 0]$, then
$\partial L/\partial W_1 = x^\top \delta_1$ and $\partial L/\partial b_1 = \sum_b \delta_{1,b}$.

In [4]:
def backward(params, cache):
    # The gradients derived above, in NumPy.
    p, h, q = cache["p"], cache["h"], cache["q"]

    g_p = 2 / B * cache["position_mask"] * (p - target)  # positions

    g_pen = 2 * lambda_o / B * cache["both"] * cache["pen"]  # overlap
    px, py = cache["pxy"][:, 0], cache["pxy"][:, 1]
    takes = np.column_stack([px <= py, px > py])  # which depth the minimum passes on
    active = (cache["a"] - cache["s"]) > 0  # the ReLU inside each depth
    g_gap = g_pen[:, None] * takes * active * (-cache["gap"] / cache["s"])  # dL/d(x1 - x2), dL/d(y1 - y2)
    g_meters = np.stack([g_gap, -g_gap], axis=1)  # item 1 gets +, item 2 gets -
    g_p = g_p + (g_meters * room[:, None, :]).reshape(B, 4)  # x = u W, y = v D

    delta2 = g_p * p * (1 - p)  # sigmoid
    onehot = np.eye(4)[rot_target]
    delta3 = (lambda_r / B * mask[..., None] * (q - onehot)).reshape(B, 8)  # softmax + cross-entropy

    g_h = delta2 @ params["W2"].T + delta3 @ params["W3"].T  # both heads read h
    delta1 = g_h * (cache["z1"] > 0)  # ReLU
    return {"W1": x.T @ delta1, "b1": delta1.sum(axis=0),
            "W2": h.T @ delta2, "b2": delta2.sum(axis=0),
            "W3": h.T @ delta3, "b3": delta3.sum(axis=0)}


manual = backward(params, cache)
{name: np.round(grad.ravel()[:3], 6) for name, grad in manual.items()}

{'W1': array([ 0.031091, -0.101659, -0.013527]),
 'b1': array([ 0.27654 , -1.196204, -0.275373]),
 'W2': array([ 2.793779, -0.010253, -2.792934]),
 'b2': array([ 12.013149,   3.840127, -12.044265]),
 'W3': array([ 0.013265,  0.034034, -0.002467]),
 'b3': array([-0.002025,  0.317748, -0.250604])}

## 3. The same loss with PyTorch autograd

The reference uses `torch.nn.functional.cross_entropy` on the logits and the project's
`geometry.penetration_depth` for the overlap, then lets autograd compute every gradient.

In [5]:
tensors = {name: torch.tensor(value, requires_grad=True) for name, value in params.items()}
X, M, T, R = (torch.tensor(a) for a in (x, mask, target, room))
RT = torch.tensor(rot_target)

z1 = X @ tensors["W1"] + tensors["b1"]
h = torch.relu(z1)
p = torch.sigmoid(h @ tensors["W2"] + tensors["b2"])
logits = (h @ tensors["W3"] + tensors["b3"]).reshape(B, 2, 4)

loss_pos = (M.repeat_interleave(2, dim=1) * (p - T) ** 2).sum() / B
ce = F.cross_entropy(logits.reshape(-1, 4), RT.reshape(-1), reduction="none").reshape(B, 2)
loss_rot = lambda_r * (M * ce).sum() / B
centers = p.reshape(B, 2, 2) * R[:, None, :]
size = torch.tensor(footprint)  # (B, item, axis)
pen = geometry.penetration_depth(centers[:, 0], size[:, 0], centers[:, 1], size[:, 1], margin=margin)
loss_ov = lambda_o * (M[:, 0] * M[:, 1] * pen ** 2).sum() / B
loss_torch = loss_pos + loss_rot + loss_ov
loss_torch.backward()

differences = {name: float(np.abs(manual[name] - tensors[name].grad.numpy()).max()) for name in params}
max_difference = max(differences.values())
print(f"loss: NumPy {loss:.12f}, PyTorch {loss_torch.item():.12f}")
for name, difference in differences.items():
    print(f"  d L / d {name}: largest difference {difference:.2e}")
assert abs(loss - loss_torch.item()) < 1e-12 and max_difference < 1e-6
print(f"PASS: largest difference {max_difference:.2e} < 1e-6")

loss: NumPy 7.850290647479, PyTorch 7.850290647479
  d L / d W1: largest difference 4.44e-16
  d L / d b1: largest difference 4.44e-16
  d L / d W2: largest difference 1.78e-15
  d L / d b2: largest difference 1.78e-15
  d L / d W3: largest difference 2.78e-17
  d L / d b3: largest difference 1.11e-16
PASS: largest difference 1.78e-15 < 1e-6


## 4. A third witness: finite differences

Autograd and the hand derivation could share a misunderstanding, so a few entries are also checked
against the definition of the derivative, $\big(L(\theta + h) - L(\theta - h)\big) / 2h$.

In [6]:
def numeric(name, index, h=1e-6):
    up, down = ({k: v.copy() for k, v in params.items()} for _ in range(2))
    up[name][index] += h
    down[name][index] -= h
    return (forward(up)[0] - forward(down)[0]) / (2 * h)


fd_error = 0.0
for name, grad in manual.items():  # the largest gradient entry of each parameter
    index = np.unravel_index(np.abs(grad).argmax(), grad.shape)
    estimate = numeric(name, index)
    fd_error = max(fd_error, abs(estimate - grad[index]) / max(abs(grad[index]), 1e-12))
    print(f"d L / d {name}{[int(i) for i in index]}: hand {grad[index]: .9f}   finite difference {estimate: .9f}")
assert fd_error < 1e-6
print(f"largest relative difference {fd_error:.1e}")

d L / d W1[4, 5]: hand  1.264756750   finite difference  1.264756750
d L / d b1[5]: hand  1.264756750   finite difference  1.264756750
d L / d W2[1, 2]: hand -7.148087089   finite difference -7.148087088
d L / d b2[2]: hand -12.044265066   finite difference -12.044265057
d L / d W3[1, 1]: hand  0.216087392   finite difference  0.216087392
d L / d b3[1]: hand  0.317747807   finite difference  0.317747807
largest relative difference 1.8e-09


## 5. Why the overlap penalty is a penetration depth (Appendix K)

A side table placed wholly inside the sofa's footprint: the exact intersection area is the table's own
area wherever it sits inside, so its gradient is zero and the optimizer gets no push at all. The
penetration depth still pushes the table out, along the shorter way out (here y).

In [7]:
sofa_c, sofa_s = torch.tensor([2.8, 3.55]), torch.tensor([2.10, 0.90])
table_c = torch.tensor([2.5, 3.40], requires_grad=True)  # wholly inside the sofa's footprint
table_s = torch.tensor([0.45, 0.45])

area = geometry.overlap_area(sofa_c, sofa_s, table_c, table_s)
(area_grad,) = torch.autograd.grad(area, table_c)
depth = geometry.penetration_depth(sofa_c, sofa_s, table_c, table_s)
(depth_grad,) = torch.autograd.grad(depth ** 2, table_c)
print(f"overlap area {area.item():.4f} m^2, gradient {area_grad.numpy()}  <- no push at all")
print(f"penetration depth {depth.item():.4f} m, gradient of its square {depth_grad.numpy()}  <- pushes the table out along y")

overlap area 0.2025 m^2, gradient [0. 0.]  <- no push at all
penetration depth 0.5250 m, gradient of its square [0.      1.04997]  <- pushes the table out along y


## Summary

| Step | Local derivative | Tech Spec |
|---|---|---|
| Squared error on positions | $2(p - p^*)$, times the mask | Appendix D, N |
| Overlap penalty | $2\,\mathrm{pen}$, through min, ReLU and $\sqrt{g^2 + \varepsilon}$, times $W$ or $D$ | Appendix K, 3.2 |
| Sigmoid head | $p(1 - p)$ | Appendix B |
| Softmax + cross-entropy | $q - \mathrm{onehot}(r^*)$, times the mask | Appendix C, N |
| Linear layer $z = hW + b$ | $h^\top\delta$, $\sum_b \delta$, $\delta W^\top$ | Appendix A, F |
| ReLU | $[z > 0]$ | Appendix E |

The hand-derived gradients agree with autograd to rounding error, far below the $10^{-6}$ pass mark.